In [1]:
from datasets import load_dataset

ds = load_dataset("coastalcph/lex_glue", "ledgar")
names = ds["train"].features["label"].names        # the 100 clause-type names
print(f"LEDGAR: {len(ds['train']):,} training clauses, {len(names)} clause types\n")

# --- a few real clauses, straight from the contracts ---
want = ["Governing Laws", "Notices", "Counterparts"]
seen = {}
for ex in ds["train"]:
    lbl = names[ex["label"]]
    if lbl in want and lbl not in seen:
        seen[lbl] = ex["text"].strip().replace("\n", " ")
    if len(seen) == len(want):
        break

for lbl in want:
    print(f"[{lbl}]  {seen[lbl][:140]} ...\n")

# --- the reframe: turn (clause, label) into ONE string a GPT can learn ---
# append the label as a single special token, e.g. "<Governing Laws>"
clause = seen["Governing Laws"]
training_string = clause + " <Governing Laws>"
print("=" * 64)
print("The reframe  (clause text  +  label as one token appended at the end):\n")
print("  clause (verbatim):")
print("     " + clause[:120] + " ...")
print("\n  training string = clause + label token  (see the tail):")
print("     ... " + clause[-55:] + "  <Governing Laws>")
print("\nIn training the model reads the clause and learns to emit that final")
print("<Governing Laws> token. At test time we feed it a NEW clause and read")
print("back whichever <...> token it generates — that generated token is the")
print("predicted clause type. Classification, done as generation.")

/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LEDGAR: 60,000 training clauses, 100 clause types

[Governing Laws]  This Agreement shall be construed and enforced in accordance with the laws of the State of Minnesota (without reference to its conflicts of  ...

[Notices]  All notices required or permitted under this Agreement will be in writing, will reference this Agreement, and will be deemed given: (i) when ...

[Counterparts]  This Amendment may be executed by one or more of the parties hereto on any number of separate counterparts, and all of said counterparts tak ...

The reframe  (clause text  +  label as one token appended at the end):

  clause (verbatim):
     This Agreement shall be construed and enforced in accordance with the laws of the State of Minnesota (without reference  ...

  training string = clause + label token  (see the tail):
     ... without reference to its conflicts of laws provisions).  <Governing Laws>

In training the model reads the clause and learns to emit that final
<Governing Laws> token. At test

In [2]:
from datasets import load_dataset
from collections import Counter
ds = load_dataset("coastalcph/lex_glue", "ledgar")
names = ds["train"].features["label"].names            # 100 clause-type names
TOP = [l for l, _ in Counter(ds["train"]["label"]).most_common(6)]   # 6 most common types
topset = set(TOP)

print("splits:", {k: len(v) for k, v in ds.items()})
print("number of clause types:", len(names))
print("\nthe 6 most common types (what we'll train on):", [names[i] for i in TOP])
print("\nsample clauses:")
for e in ds["train"].select(range(3)):
    print(f"  [{names[e['label']]}]  {e['text'][:90]}...")

splits: {'train': 60000, 'test': 10000, 'validation': 10000}
number of clause types: 100

the 6 most common types (what we'll train on): ['Governing Laws', 'Notices', 'Counterparts', 'Entire Agreements', 'Severability', 'Survival']

sample clauses:
  [Waivers]  Except as otherwise set forth in this Debenture, the Company, for itself and its legal rep...
  [Erisa]  No ERISA Event has occurred or is reasonably expected to occur that, when taken together w...
  [Counterparts]  This Amendment may be executed by one or more of the parties hereto on any number of separ...


In [4]:
from datasets import load_dataset
from collections import Counter
ds = load_dataset("coastalcph/lex_glue", "ledgar")
names = ds["train"].features["label"].names            # 100 clause-type names
TOP = [l for l, _ in Counter(ds["train"]["label"]).most_common(6)]   # 6 most common types
topset = set(TOP)
import random
from collections import defaultdict

# a BALANCED subset: equal clauses per type, so the model can't win by always guessing the commonest label
def balanced_subset(split, per_class, seed):
    picked = defaultdict(list)
    for i, lab in enumerate(split["label"]):
        if lab in topset and len(picked[lab]) < per_class:
            picked[lab].append(i)
        if all(len(picked[l]) >= per_class for l in TOP):
            break
    flat = [i for l in TOP for i in picked[l]]
    random.Random(seed).shuffle(flat)
    return split.select(flat)

train = balanced_subset(ds["train"], per_class=100, seed=0)   # 600 clauses, 100 per type
def reframe(e):
    return e["text"].strip() + " <" + names[e["label"]] + ">"

print("reframed training examples:\n")
for e in train.select(range(4)):
    print(" ", reframe(e)[:110], "...")
print("\ntraining subset:", len(train), "clauses  (", len(TOP), "types x 100, balanced )")

reframed training examples:

  Any provision of this Third Amendment that is prohibited or unenforceable in any jurisdiction shall, as to suc ...
  Unless specifically provided herein, this Agreement, the Severance Agreement, and the Employment Agreement con ...
  Any required notices hereunder shall be given in writing and sent by facsimile or priority mail to the address ...
  This Agreement and the Shareholder Agreement represent the entire agreement of the parties with respect to the ...

training subset: 600 clauses  ( 6 types x 100, balanced )


In [5]:
from datasets import load_dataset
from collections import Counter
ds = load_dataset("coastalcph/lex_glue", "ledgar")
names = ds["train"].features["label"].names            # 100 clause-type names
TOP = [l for l, _ in Counter(ds["train"]["label"]).most_common(6)]   # 6 most common types
topset = set(TOP)
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("distilgpt2")
label_tokens = [f"<{names[i]}>" for i in TOP]

demo = label_tokens[0]
print(f"BEFORE: {demo!r} -> {len(tok.tokenize(demo))} tokens:", tok.tokenize(demo))

tok.add_tokens(label_tokens)                 # 1. add the new tokens
tok.pad_token = tok.eos_token                # 3. give GPT-2 a pad token

print(f"AFTER : {demo!r} -> {len(tok.tokenize(demo))} token :", tok.tokenize(demo))

model = AutoModelForCausalLM.from_pretrained("distilgpt2")
old = model.get_input_embeddings().weight.shape[0]
model.resize_token_embeddings(len(tok))      # 2. grow the embeddings to match
new = model.get_input_embeddings().weight.shape[0]
print(f"\nembeddings resized: {old} -> {new} rows (+{new - old}, one per added label token)")

BEFORE: '<Governing Laws>' -> 6 tokens: ['<', 'G', 'over', 'ning', 'ĠLaws', '>']
AFTER : '<Governing Laws>' -> 1 token : ['<Governing Laws>']


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 7153.82it/s]
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`



embeddings resized: 50257 -> 50263 rows (+6, one per added label token)


In [6]:
# any legal word GPT-2 never saw is still representable — as a sequence of known subword pieces:
for w in ["indemnification", "hereinafter", "notwithstanding", "Delaware", "zzxqwv"]:
    print(f"{w:16s} -> {tok.tokenize(w)}")

print("\nNothing is 'missing' — BPE can spell ANY string out of its ~50k pieces, so an unseen")
print("word just costs a few extra tokens. That's why we add no vocabulary words. The six")
print("<clause type> tokens are added for a different reason: we want each clause TYPE to be one")
print("atomic token with its own learnable embedding — a single category to predict in one step,")
print("not a phrase the model has to spell out piece by piece (and could get wrong midway).")

indemnification  -> ['ind', 'emn', 'ification']
hereinafter      -> ['here', 'ina', 'fter']
notwithstanding  -> ['not', 'withstanding']
Delaware         -> ['Del', 'aware']
zzxqwv           -> ['zz', 'x', 'q', 'w', 'v']

Nothing is 'missing' — BPE can spell ANY string out of its ~50k pieces, so an unseen
word just costs a few extra tokens. That's why we add no vocabulary words. The six
<clause type> tokens are added for a different reason: we want each clause TYPE to be one
atomic token with its own learnable embedding — a single category to predict in one step,
not a phrase the model has to spell out piece by piece (and could get wrong midway).


In [7]:
batch = [
    "This Agreement shall be governed by the laws of the State of Delaware.",   # longer
    "Confidential Information stays secret.",                                   # shorter
]
enc = tok(batch, padding=True, return_tensors="pt")

print("batch shape (rows x tokens):", tuple(enc["input_ids"].shape), "\n")
for i, text in enumerate(batch):
    ids  = enc["input_ids"][i].tolist()
    mask = enc["attention_mask"][i].tolist()
    print(f"clause {i}: {len(ids)} slots, {mask.count(0)} of them padding")
    print("  ids :", ids)
    print("  mask:", mask, " <- 0 = padding, ignored by the model\n")

print(f"pad token = {tok.pad_token!r} (id {tok.pad_token_id}), reused from end-of-text.")
print("Both rows are now the same length, so they stack into ONE tensor the GPU runs in parallel.")
print("In Lesson 5 we pad every clause to a fixed length up front, and mark the padded")
print("positions as -100 in the labels so they never contribute to the loss.")

batch shape (rows x tokens): (2, 14) 

clause 0: 14 slots, 0 of them padding
  ids : [1212, 12729, 2236, 307, 21825, 416, 262, 3657, 286, 262, 1812, 286, 19603, 13]
  mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]  <- 0 = padding, ignored by the model

clause 1: 14 slots, 8 of them padding
  ids : [18546, 35599, 6188, 14768, 3200, 13, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256]
  mask: [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]  <- 0 = padding, ignored by the model

pad token = '<|endoftext|>' (id 50256), reused from end-of-text.
Both rows are now the same length, so they stack into ONE tensor the GPU runs in parallel.
In Lesson 5 we pad every clause to a fixed length up front, and mark the padded
positions as -100 in the labels so they never contribute to the loss.


In [8]:
from datasets import load_dataset
from collections import Counter
ds = load_dataset("coastalcph/lex_glue", "ledgar")
names = ds["train"].features["label"].names            # 100 clause-type names
TOP = [l for l, _ in Counter(ds["train"]["label"]).most_common(6)]   # 6 most common types
topset = set(TOP)
import random, torch
from collections import defaultdict
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          TrainingArguments, Trainer, default_data_collator)

def balanced_subset(split, per_class, seed):
    picked = defaultdict(list)
    for i, lab in enumerate(split["label"]):
        if lab in topset and len(picked[lab]) < per_class:
            picked[lab].append(i)
        if all(len(picked[l]) >= per_class for l in TOP):
            break
    flat = [i for l in TOP for i in picked[l]]
    random.Random(seed).shuffle(flat)
    return split.select(flat)

train = balanced_subset(ds["train"], per_class=100, seed=0)

tok = AutoTokenizer.from_pretrained("distilgpt2"); tok.pad_token = tok.eos_token
tok.add_tokens([f"<{names[i]}>" for i in TOP])
model = AutoModelForCausalLM.from_pretrained("distilgpt2"); model.resize_token_embeddings(len(tok))

def encode(e):
    label_str = "<" + names[e["label"]] + ">"
    enc = tok(e["text"].strip() + " " + label_str, truncation=True, padding="max_length", max_length=96)
    label_tid = tok.convert_tokens_to_ids(label_str)
    labels = [-100] * len(enc["input_ids"])            # ignore every position ...
    if label_tid in enc["input_ids"]:
        labels[enc["input_ids"].index(label_tid)] = label_tid   # ... except the label token
    enc["labels"] = labels
    return enc

tokenized = train.map(encode, remove_columns=train.column_names)
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

args = TrainingArguments(output_dir="legalgpt-ft", num_train_epochs=3, per_device_train_batch_size=8,
                         learning_rate=5e-4, logging_steps=50, report_to="none", save_strategy="no",
                         disable_tqdm=True)
trainer = Trainer(model=model, args=args, train_dataset=tokenized, data_collator=default_data_collator)
trainer.train()

trainer.save_model("legalgpt-ft"); tok.save_pretrained("legalgpt-ft")
print("\nsaved fine-tuned model to legalgpt-ft/  (Lessons 6-8 load it)")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 3964.86it/s]
/opt/conda/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


{'loss': '4.16', 'grad_norm': '228.1', 'learning_rate': '0.0003911', 'epoch': '0.6667'}
{'loss': '1.921', 'grad_norm': '95.59', 'learning_rate': '0.00028', 'epoch': '1.333'}
{'loss': '1.802', 'grad_norm': '114.2', 'learning_rate': '0.0001689', 'epoch': '2'}
{'loss': '1.744', 'grad_norm': '107.5', 'learning_rate': '5.778e-05', 'epoch': '2.667'}
{'train_runtime': '497.4', 'train_samples_per_second': '3.619', 'train_steps_per_second': '0.452', 'train_loss': '2.329', 'epoch': '3'}


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.01s/it]


saved fine-tuned model to legalgpt-ft/  (Lessons 6-8 load it)


In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from collections import Counter

ds = load_dataset("coastalcph/lex_glue", "ledgar")
names = ds["train"].features["label"].names
TOP = [l for l, _ in Counter(ds["train"]["label"]).most_common(6)]

tok = AutoTokenizer.from_pretrained("legalgpt-ft")     # your fine-tuned model from Lesson 5
model = AutoModelForCausalLM.from_pretrained("legalgpt-ft").eval()
label_id_to_name = {tok.convert_tokens_to_ids(f"<{names[i]}>"): names[i] for i in TOP}
label_ids = list(label_id_to_name)

@torch.no_grad()
def classify(clause):
    "Ask the model for the next token after the clause, restricted to the label tokens."
    logits = model(**tok(clause.strip() + " ", return_tensors="pt")).logits[0, -1]
    best = max(label_ids, key=lambda t: logits[t].item())
    return label_id_to_name[best]

# tag a few held-out validation clauses
val = ds["validation"].filter(lambda e: e["label"] in set(TOP)).shuffle(seed=1).select(range(6))
print(f"{'TRUE':22s}{'PREDICTED':22s}CLAUSE")
for e in val:
    print(f"{names[e['label']]:22s}{classify(e['text']):22s}{e['text'][:60]}...")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 5998.85it/s]


TRUE                  PREDICTED             CLAUSE
Counterparts          Counterparts          This Agreement may be executed in counterparts (and by diffe...
Severability          Counterparts          Should any clause, sentence, paragraph, subsection, Section ...
Entire Agreements     Governing Laws        This Agreement constitutes the complete understanding betwee...
Survival              Governing Laws        The rights and obligations of the parties under the provisio...
Survival              Counterparts          All covenants, agreements, representations and warranties ma...
Entire Agreements     Governing Laws        This Agreement constitutes the entire understanding of the p...


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from collections import Counter

ds = load_dataset("coastalcph/lex_glue", "ledgar")
names = ds["train"].features["label"].names
TOP = [l for l, _ in Counter(ds["train"]["label"]).most_common(6)]

tok = AutoTokenizer.from_pretrained("legalgpt-ft")     # your fine-tuned model from Lesson 5
model = AutoModelForCausalLM.from_pretrained("legalgpt-ft").eval()
label_id_to_name = {tok.convert_tokens_to_ids(f"<{names[i]}>"): names[i] for i in TOP}
label_ids = list(label_id_to_name)

@torch.no_grad()
def classify(clause):
    "Ask the model for the next token after the clause, restricted to the label tokens."
    logits = model(**tok(clause.strip() + " ", return_tensors="pt")).logits[0, -1]
    best = max(label_ids, key=lambda t: logits[t].item())
    return label_id_to_name[best]

# tag a few held-out validation clauses
val = ds["validation"].filter(lambda e: e["label"] in set(TOP)).shuffle(seed=1).select(range(6))
print(f"{'TRUE':22s}{'PREDICTED':22s}CLAUSE")
for e in val:
    print(f"{names[e['label']]:22s}{classify(e['text']):22s}{e['text'][:60]}...")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 2185.51it/s]


TRUE                  PREDICTED             CLAUSE
Counterparts          Counterparts          This Agreement may be executed in counterparts (and by diffe...
Severability          Counterparts          Should any clause, sentence, paragraph, subsection, Section ...
Entire Agreements     Governing Laws        This Agreement constitutes the complete understanding betwee...
Survival              Governing Laws        The rights and obligations of the parties under the provisio...
Survival              Counterparts          All covenants, agreements, representations and warranties ma...
Entire Agreements     Governing Laws        This Agreement constitutes the entire understanding of the p...


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from collections import Counter

ds = load_dataset("coastalcph/lex_glue", "ledgar")
names = ds["train"].features["label"].names
TOP = [l for l, _ in Counter(ds["train"]["label"]).most_common(6)]

tok = AutoTokenizer.from_pretrained("legalgpt-ft")     # your fine-tuned model from Lesson 5
model = AutoModelForCausalLM.from_pretrained("legalgpt-ft").eval()
label_id_to_name = {tok.convert_tokens_to_ids(f"<{names[i]}>"): names[i] for i in TOP}
label_ids = list(label_id_to_name)

@torch.no_grad()
def classify(clause):
    "Ask the model for the next token after the clause, restricted to the label tokens."
    logits = model(**tok(clause.strip() + " ", return_tensors="pt")).logits[0, -1]
    best = max(label_ids, key=lambda t: logits[t].item())
    return label_id_to_name[best]

val = ds["validation"].filter(lambda e: e["label"] in set(TOP)).shuffle(seed=2).select(range(120))
preds = [classify(e["text"]) for e in val]
trues = [names[e["label"]] for e in val]
acc = sum(p == t for p, t in zip(preds, trues)) / len(val)
print(f"clause-type accuracy on {len(val)} held-out clauses: {acc:.1%}   (random = {1/len(TOP):.1%})")

print("\na few misses (note how the confused types tend to be similar boilerplate):")
shown = 0
for p, t, e in zip(preds, trues, val):
    if p != t and shown < 5:
        print(f"  true={t:20s} pred={p:20s} :: {e['text'][:55]}...")
        shown += 1

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 10588.86it/s]


clause-type accuracy on 120 held-out clauses: 21.7%   (random = 16.7%)

a few misses (note how the confused types tend to be similar boilerplate):
  true=Counterparts         pred=Governing Laws       :: Administrative Agent shall have received from the Lende...
  true=Counterparts         pred=Governing Laws       :: This Letter Agreement may be executed in one or more co...
  true=Survival             pred=Governing Laws       :: Unless this Agreement is terminated under Section 8 , t...
  true=Survival             pred=Governing Laws       :: Without prejudice to the survival of any other agreemen...
  true=Governing Laws       pred=Counterparts         :: All questions concerning the construction, validity, en...


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from collections import Counter

ds = load_dataset("coastalcph/lex_glue", "ledgar")
names = ds["train"].features["label"].names
TOP = [l for l, _ in Counter(ds["train"]["label"]).most_common(6)]

class LegalClauseTagger:
    "Load the fine-tuned model and tag a contract clause with its type."
    def __init__(self, path="legalgpt-ft"):
        self.tok = AutoTokenizer.from_pretrained(path)
        self.model = AutoModelForCausalLM.from_pretrained(path).eval()
        self.label_id_to_name = {self.tok.convert_tokens_to_ids(f"<{names[i]}>"): names[i] for i in TOP}

    @torch.no_grad()
    def classify_clause(self, clause):
        logits = self.model(**self.tok(clause.strip() + " ", return_tensors="pt")).logits[0, -1]
        best = max(self.label_id_to_name, key=lambda t: logits[t].item())
        return self.label_id_to_name[best]

tagger = LegalClauseTagger()
for clause in [
    "This Agreement shall be governed by and construed in accordance with the laws of the State of New York.",
    "All notices under this Agreement shall be in writing and sent by certified mail to the addresses below.",
    "This Agreement may be executed in counterparts, each of which shall be deemed an original.",
]:
    print(f"{tagger.classify_clause(clause):22s} :: {clause[:62]}...")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 21342.20it/s]


Governing Laws         :: This Agreement shall be governed by and construed in accordanc...
Governing Laws         :: All notices under this Agreement shall be in writing and sent ...
Governing Laws         :: This Agreement may be executed in counterparts, each of which ...
